#BASIC

In [0]:
# 1
rows = [
    [1, 101, "Laptop", "Electronics", 55000, "2026-01-05", "sakshi@gmail.com"],
    [2, 102, "Phone", "Electronics", 30000, "2026-01-07", "nishi@gmail.com"],
    [3, 103, "Chair", "Furniture", 4500, "2026-01-10", None],
    [4, 104, "Table", "Furniture", None, "2026-01-12", "neha@gmail.com"],
    [5, 105, "Headphones", "Electronics", 2500, "2026-01-15", "amit@gmail.com"],
    [6, 106, "Shoes", "Fashion", 3200, "2026-01-18", None],
    [7, 107, "Watch", "Fashion", 5000, "2026-01-20", "pooja@gmail.com"],
    [8, 108, "Keyboard", "Electronics", 1800, "2026-01-22", "rohit@gmail.com"],
    [8, 108, "Keyboard", "Electronics", 1800, "2026-01-22", "rohit@gmail.com"],
    [9, 109, "Backpack", "Fashion", 2200, "2026-01-25", None],
    [10, 110, "Monitor", "Electronics", 15000, "2026-01-27", "simran@gmail.com"],
    [11, 111, "Sofa", "Furniture", 25000, "2026-02-01", "karan@gmail.com"],
    [12, 112, "Mouse", "Electronics", 900, "2026-02-03", None],
    [13, 113, "Jacket", "Fashion", 4200, "2026-02-05", "meena@gmail.com"],
    [13, 113, "Jacket", "Fashion", 4200, "2026-02-05", "meena@gmail.com"],
    [14, 114, "Desk", "Furniture", 8000, "2026-02-08", "arjun@gmail.com"],
    [15, 115, "Tablet", "Electronics", 22000, "2026-02-10", None]
]

columns = ["order_id", "customer_id", "product", "category", "amount", "order_date", "email"]
df = spark.createDataFrame(rows, columns)
df = df.write.mode("overwrite").saveAsTable("cyntexa_dev.sales.ecommerce")



In [0]:
from pyspark.sql.functions import *

df = df.filter(col("amount").isNotNull()).distinct()
df = df.dropDuplicates(["order_id"])
df.show()

In [0]:
# 2 - column rename

df = df.withColumnsRenamed({
    "product" : "product_name",
    "amount" : "price",
    "order_date" : "date"
})

display(df)

- 3

Connect a Databricks Repo to a Git provider and make your first commit of a cleaning notebook.

1. Create git folder
2. write code and then add chnages 
3. push the notebook in main branch.

In [0]:
# 4 
from pyspark.sql.functions import col

df = spark.read.table("cyntexa_dev.sales.ecommerce")
df = df.dropna(subset=["amount"])
df = df.dropna(how="all")
df = df.fillna({
    "email": "unknown"
})
df = df.dropDuplicates()
df = df.orderBy("order_date")

display(df)

In [0]:
# 5
from pyspark.sql.functions import sum, col

revenue_by_category = (
    df
    .groupBy("category")
    .agg(
        sum("amount").alias("total_revenue")
    )
)
customer_df = spark.read.table("cyntexa_dev.sales.customers")
joined_df = df.join(
    customer_df,
    on="customer_id",
    how="left"
)

display(joined_df)
display(revenue_by_category)

6.
- Change - add order by date
- why - to see the latest order


#ADVANCED

In [0]:
# 7
from pyspark.sql import functions as F

df = spark.read.table("cyntexa_dev.sales.ecommerce")


df = df.withColumn(
    "order_date_clean",
    F.coalesce(
        F.to_date("order_date", "yyyy-MM-dd"),
        F.to_date("order_date", "dd/MM/yyyy"),
        F.to_date("order_date", "yyyy/MM/dd")
    )
)

invalid_df = df.filter(
    F.col("amount").isNull() |
    F.col("order_date_clean").isNull()
)

print("Invalid records:")
invalid_df.show()

clean_df = (
    df
    .select(
        "order_id",
        "customer_id",
        "product",
        "category",
        "amount",
        F.col("order_date_clean").alias("order_date"),
        "email"
    )
)

clean_df = clean_df.withColumn(
    "email",
    F.when(
        (F.col("email").isNull()) |
        (F.lower(F.col("email")) == "unknown"),
        None
    ).otherwise(F.col("email"))
)

clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("cyntexa_dev.sales.orders_clean")




Approach - 

1. create a datafram from a existing table
2. make the same data format
3. find invalid records
4. clean that dataframe
5. save the cleaned df as a table.

8.

branching strategy - 
- Developers create feature branches from dev for development and never push directly to main. 
- After completing a feature, the developer pushes the branch and opens a PR into dev. 
- The developer must test the changes and provide a clear PR description. 
- Once the PR is approved and checks pass, it can be merged into dev. 
- After integration testing, a PR from dev to main is created. main is protected and represents production-ready code, so direct pushes are not allowed.

In [0]:
%sql
-- 9
SELECT 
  category,
  COUNT(customer_id) as total_customer
FROM cyntexa_dev.sales.orders_clean 
GROUP BY category
ORDER BY total_customer DESC
LIMIT 1

In [0]:
%sql
-- month-over-month growth,
SELECT month(order_date) as month,SUM(amount) as total_revenue FROM cyntexa_dev.sales.orders_clean GROUP BY month(order_date)

In [0]:
%sql
-- average order value trend
SELECT month(order_date) as month,round(AVG(amount),2) as avg_order_value FROM cyntexa_dev.sales.orders_clean GROUP BY month(order_date);
    